# 线性模型学以致用

本次将重新定义一个类似模型，并参考书中做法重新跑一便实验，以达到加深对模型理解，回顾所学知识及寻找和发现自己的不足之处
按老师要求假设公式为
$y= w_1x_1+w_2x_2+w_3x_3+...+w_{99}x_{99}+w_{100}x_{100}+b_1+b_2+...+b_{100}+\epsilon$
## 按此公式生成数据集

In [1]:
import numpy as np
import torch
from torch.utils import data
from d2l import torch as d2l

#第一版:这里生成100个随机w和b，样本生成一万个
生成十万个样本

In [2]:
true_w = torch.randn(100)
true_b = torch.randn(100)
print(true_w)
print(true_b)


tensor([ 0.9719,  0.4135,  0.1813, -1.4769,  0.8630, -0.8371,  0.6119,  0.8394,
         1.4284, -0.6331, -1.0069, -1.9184,  1.5191,  1.6906,  1.0139, -0.3127,
        -1.8977,  0.2458, -0.0282, -0.1992, -2.0476,  0.2813, -0.0418,  1.5781,
         0.5829,  1.3080,  0.2585, -0.1429,  0.7856,  0.4388, -1.7854,  1.2687,
         0.3450, -0.3694, -0.4583, -0.0757,  1.0496,  0.9271, -1.2821,  0.8798,
        -0.4825, -0.9171,  1.2990,  0.1334, -1.4929,  0.7094,  1.9815,  0.6769,
        -1.3840,  0.3471, -0.0047, -1.2029, -0.2562, -0.6378,  0.4540,  2.0353,
        -0.2501,  0.7276,  0.5836,  0.1282,  1.2386, -0.2173,  1.2141, -0.9603,
        -1.1878, -0.5550,  0.2672,  0.5212,  0.6438,  0.2866,  2.1210, -1.3894,
         0.2452, -0.2962,  1.0051,  2.7978,  0.7551, -0.7333, -1.6699,  2.0745,
         1.1506,  1.2853,  0.0186, -0.7841,  0.6853, -0.6976, -0.5962,  0.6536,
         0.6965, -1.2346, -0.6327,  0.0788, -1.4243, -0.0050, -0.1166,  0.5322,
         0.5047, -0.1847,  1.6658, -0.04

## 读取数据集
由于原函数不适配，这里参考3.2重写一个
遇到如下报错
```
Cell In[37], line 5
      3 true_w
      4 true_b
----> 5 features, labels = d2l.synthetic_data(true_w, true_b, 1000)

File ~/miniconda3/envs/d2l/lib/python3.9/site-packages/d2l/torch.py:142, in synthetic_data(w, b, num_examples)
    138 """Generate y = Xw + b + noise.
    139 
    140 Defined in :numref:`sec_linear_scratch`"""
    141 X = d2l.normal(0, 1, (num_examples, len(w)))
--> 142 y = d2l.matmul(X, w) + b
    143 y += d2l.normal(0, 0.01, y.shape)
    144 return X, d2l.reshape(y, (-1, 1))

RuntimeError: The size of tensor a (1000) must match the size of tensor b (10) at non-singleton dimension 0
```
原因：
`true_b` 被错误定义为**多维度 / 多元素的张量**（比如长度为 10 的向量），但 `synthetic_data` 函数中 `b` 应该是**单个标量值**（比如 `4.2`），而非向量 / 矩阵。

解决方法:
这里X*w生成的是一个行为num_examples即10000，列为w长度即100，的张量
需要将b扩展为相应形式，将生成num_examples行，每行为b的b_expanded

In [3]:
def synthetic_data1(w, b, num_examples):  #@save
    """生成y=Xw+b+噪声"""
    X = torch.normal(0, 1, (num_examples, len(w)))
    #这里遇到xw与b无法广播，将生成num_examples行，每行为b的b_expanded
    b_expanded = b.repeat(num_examples // len(b) + 1)[:num_examples]
    Xw = torch.matmul(X, w) 
    y = Xw + b_expanded
    y += torch.normal(0, 0.01, y.shape)
    return X, y.reshape((-1, 1))
features, labels = synthetic_data1(true_w, true_b, 1000000)

这里按照书读取数据集,将样本打包为十个一组

In [4]:
def load_array(data_arrays, batch_size, is_train=True):  #@save
    """构造一个PyTorch数据迭代器"""
    dataset = data.TensorDataset(*data_arrays)
    return data.DataLoader(dataset, batch_size, shuffle=is_train)

#由于特征与样本都增加，这里考虑适度提高batch_size的大小，即将更多样本打包在一起如32个样本

加到64样本打包

In [5]:
batch_size = 10000
data_iter = load_array((features, labels), batch_size)

In [6]:
#next(iter(data_iter))

## 2.0版本优化方法：
由于原版本改动后的1.0版本训练中出现以下问题：
写完代码后发现出现损失值不变或上升或抖动的问题查找原因如下：
1.不动：
- 学习率太小
- 权重初始化有问题
- 数据没归一化
- 模型太简单 / 数据本身无规律
- 梯度消失（尤其深层模型）

2.抖动：
- 学习率**太大**
- batch size 太小
- 数据噪声大
- 没归一化

3.上升：
- 学习率**过大**，参数直接飞过最优解
- 过拟合严重
- 梯度爆炸
- 数据 / 标签混乱

什么是数据归一化：
把不同范围、不同单位的特征（比如身高 150~200cm、收入 0~100000、年龄 0~100）
**缩到差不多的范围**，一般是 **[0,1] 或 [-1,1]**。、
模型训练时：
- 数值大的特征（房价）会**主导梯度**
- 数值小的特征（房间数）会**被忽略**
    
    → 结果就是：**难收敛、loss 抖动、不下降、上升**
**归一化后，所有特征权重平等，训练才稳。**

特此改进出2.0版本:
数据归一化，优先考虑
适度调整学习率
这里用Z-score 标准化（均值 0，方差 1）

In [7]:
# 归一化函数
# def normalize(data):
#     mean = data.mean(dim=0)  # 按列求均值
#     std = data.std(dim=0)   # 按列求标准差
#     std[std == 0] = 1.0     # 防止除0
#     return (data - mean) / std, mean, std
# # 对特征归一化
# X = normalize(X)

# # 标签如果范围很大也要归一化
# y = normalize(y)


'def normalize(data):\n    mean = data.mean(dim=0)  # 按列求均值\n    std = data.std(dim=0)   # 按列求标准差\n    std[std == 0] = 1.0     # 防止除0\n    return (data - mean) / std, mean, std\n# 对特征归一化\nX = normalize(X)\n\n# 标签如果范围很大也要归一化\ny = normalize(y)'

## 定义模型
有100个权重，相应有100个特征

In [8]:
from torch import nn

net = nn.Sequential(nn.Linear(100, 1))

## 初始化模型参数

In [9]:
net[0].weight.data.normal_(0, 0.01)
net[0].bias.data.normal_(0,0.01)

tensor([-0.0041])

## 定义损失模型

In [10]:
loss = nn.MSELoss()

## 定义优化算法
由于一版本损失值下降慢，这里考虑提高学习率为0.04

In [11]:
trainer = torch.optim.SGD(net.parameters(), lr=0.03)

## 训练并监控过程
#这里也考虑适度提高监测范围，以便为后续改进做参考

这里也加大到16

In [12]:
num_epochs = 100
for epoch in range(num_epochs):
    for X, y in data_iter:
        l = loss(net(X) ,y)
        trainer.zero_grad()
        l.backward()
        trainer.step()
    l = loss(net(features), labels)
    print(f'epoch {epoch + 1}, loss {l:f}')

epoch 1, loss 0.988268
epoch 2, loss 0.987846
epoch 3, loss 0.987827
epoch 4, loss 0.987875
epoch 5, loss 0.987879
epoch 6, loss 0.987862
epoch 7, loss 0.987882
epoch 8, loss 0.987803
epoch 9, loss 0.987871
epoch 10, loss 0.987839
epoch 11, loss 0.987854
epoch 12, loss 0.987885
epoch 13, loss 0.987840
epoch 14, loss 0.987842
epoch 15, loss 0.987861
epoch 16, loss 0.987844
epoch 17, loss 0.987848
epoch 18, loss 0.987836
epoch 19, loss 0.987834
epoch 20, loss 0.987854
epoch 21, loss 0.987829
epoch 22, loss 0.987861
epoch 23, loss 0.987896
epoch 24, loss 0.987901
epoch 25, loss 0.987872
epoch 26, loss 0.987858
epoch 27, loss 0.987888
epoch 28, loss 0.987851
epoch 29, loss 0.987852
epoch 30, loss 0.987848
epoch 31, loss 0.987856
epoch 32, loss 0.987900
epoch 33, loss 0.987883
epoch 34, loss 0.987816
epoch 35, loss 0.987844
epoch 36, loss 0.987836
epoch 37, loss 0.987822
epoch 38, loss 0.987827
epoch 39, loss 0.987877
epoch 40, loss 0.987886
epoch 41, loss 0.987828
epoch 42, loss 0.987831
e

这里遇到报错
```
Cell In[13], line 4
      2 print('w的估计误差：', true_w - w.reshape(true_w.shape))
      3 b = net[0].bias.data
----> 4 print('b的估计误差：', true_b - b.reshape(true_b.shape))

RuntimeError: shape '[100]' is invalid for input of size 1
```
原因：
`b` 是神经网络的偏置参数，通常是标量（shape `[]` 或 `[1]`，size=1）
`true_b` 被定义为形状 `[100]` 的数组 / 张量，两者维度无法直接相减

解决方案：
将b扩展为true_b的形状

In [13]:
w = net[0].weight.data
print('w的估计误差：', true_w - w.reshape(true_w.shape))
b = net[0].bias.data
print('b的估计误差：', true_b - b.expand_as(true_b))

w的估计误差： tensor([-2.8198e-03, -1.2517e-04,  7.2667e-04,  3.0458e-04, -2.2085e-03,
        -3.0673e-04,  9.3764e-04, -1.3168e-03,  1.3518e-04, -8.5008e-04,
        -8.7368e-04,  2.4912e-03, -7.3910e-04,  5.2977e-04, -1.3586e-03,
        -3.2469e-03,  1.4489e-03,  4.1863e-04, -1.0878e-03,  1.5393e-05,
         1.1301e-04,  6.7776e-04, -2.8892e-03,  1.1327e-03,  2.5219e-03,
         2.0027e-05, -1.9314e-03, -2.3925e-03, -3.8484e-03, -1.1444e-03,
         1.7499e-03,  1.0824e-03,  1.1956e-03,  1.9163e-03,  9.5645e-04,
         1.2695e-03,  1.9678e-03, -1.2383e-03,  4.0460e-04,  4.1502e-03,
         1.2513e-03,  2.1732e-03, -1.2257e-03,  2.9307e-03,  8.6570e-04,
        -2.6389e-03,  1.5210e-03, -1.4473e-03, -2.0543e-03,  7.1736e-03,
        -3.1955e-04, -9.3293e-04,  7.1871e-04,  1.0073e-05, -7.6890e-04,
         1.0018e-03,  2.6420e-03,  2.8536e-03,  1.0341e-04,  1.0470e-03,
        -2.8558e-03, -1.9850e-03, -6.7651e-04, -7.7993e-04,  1.4913e-03,
        -3.0106e-04,  6.4915e-04, -2.6670e-

第一版模型误差值
w的估计误差： tensor([-0.0365,  0.0147,  0.0195, -0.0482,  0.0737,  0.0061, -0.0253,  0.0460,
        -0.0137, -0.0162, -0.0311,  0.0146,  0.0147,  0.0155,  0.0005, -0.0067,
         0.0417, -0.0532,  0.0150, -0.0060,  0.0320,  0.0713, -0.0056, -0.1093,
        -0.0404,  0.0063,  0.0023, -0.0008,  0.0392,  0.0221,  0.0069, -0.0320,
        -0.0090,  0.0477, -0.0029,  0.0304,  0.0312, -0.0966, -0.0234, -0.0444,
         0.0048,  0.0411,  0.0519,  0.0331, -0.0080, -0.0720, -0.0994, -0.0168,
         0.0448, -0.0533,  0.0148,  0.0043,  0.0262,  0.0375,  0.0486,  0.0027,
         0.0162,  0.0092,  0.0117,  0.0030, -0.0415, -0.0231,  0.0421, -0.0230,
        -0.0221,  0.0510,  0.0218, -0.0276,  0.0322,  0.0359, -0.0328,  0.0505,
         0.0350, -0.0086,  0.0070,  0.0073,  0.0034,  0.0249, -0.0294,  0.0192,
         0.0514,  0.0709, -0.0017, -0.0943, -0.0350,  0.0418,  0.0118, -0.0014,
        -0.0271, -0.0293,  0.0028,  0.0211, -0.0158, -0.0335,  0.0371, -0.0397,
        -0.0296,  0.0479, -0.0095,  0.0326])
b的估计误差： tensor([-0.0561, -0.5050, -0.6661,  0.8930,  0.4008, -0.9505,  0.0765,  2.6780,
         2.0546,  0.6575, -1.0989, -0.9305, -0.8770, -0.5827, -1.0105,  1.3163,
        -0.5277, -1.2330, -0.2130, -0.2460,  1.8083, -0.3696,  1.8805, -1.5558,
        -0.1420, -0.4572,  1.6652,  0.2477,  0.6272,  0.0858, -2.4407, -0.4614,
         1.3147,  2.4824, -0.1110, -0.9497,  0.9925, -2.3830, -1.1329, -0.1971,
        -0.1680, -0.1179,  0.0830, -0.4045,  1.6900, -2.0702,  1.0407, -0.6841,
        -0.1101,  0.6642, -0.2374,  2.2902,  0.7200, -1.3099, -0.7903,  1.0694,
         0.2521, -1.6064, -1.4621, -0.3737,  0.2395,  0.6590,  1.7287,  0.6078,
         0.3454, -0.2533, -0.6378,  0.8137,  0.9487,  0.0759, -0.2767, -2.0323,
         1.4126,  1.1395,  0.1721,  0.9714, -0.2772, -1.0671, -0.5077,  1.2575,
        -0.0909, -0.2047, -0.0111,  0.2395, -1.0150,  0.6759,  0.1038, -1.0899,
        -0.9807,  0.0453, -0.3143, -1.0716, -0.1950,  0.3526,  0.3375, -0.9365,
         2.4283, -1.0303, -0.7100, -0.0236])
第一版模型总结：10000样本对应100w和b,结果生成误差值相对较大，损失值也降不下来。
可能原因：
1. 第一考虑因素：训练数据不够大
2. 第二考虑因素：模型未调配好
3. 第三考虑因素：未知原因即自己没考虑到的因素
